In [8]:
!pip install --upgrade pip setuptools wheel
!pip install "numpy<2" "pandas<2.2" "scipy<1.11" "scikit-learn<1.4" "lightgbm==3.3.5"
!pip install pycaret==3.3.2

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
  Using cached scipy-1.10.1-cp39-cp39-macosx_12_0_arm64.whl.metadata (53 kB)
  Using cached scikit_learn-1.3.2-cp39-cp39-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached lightgbm-3.3.5.tar.gz (1.5 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Using cached scipy-1.10.1-cp39-cp39-macosx_12_0_arm64.whl (28.9 MB)
Using cached scikit_learn-1.3.2-cp39-cp39-macosx_12_0_arm64.whl (9.5 MB)
  error: subprocess-exited-with-error
  
  × Building wheel for lightgbm (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [104 lines of output]
      /private/var/folders/y0/9zbck24x5017gh20l_rnx3wr0000gn/T/pip-build-env-4i0vide1/overlay/lib/python3.9/site-packages/setuptools/dist.py:759: SetuptoolsDeprecationWarning: License classifiers are d

In [21]:
import pandas as pd
from pycaret.regression import *
from sklearn.model_selection import train_test_split

In [31]:
SEED = 128
target = "england_wales_demand"

df = pd.read_csv("../Dataset2_Demand/6_Elec_Demand_Final.csv")

In [32]:
df["settlement_date"] = pd.to_datetime(df["settlement_date"])
df["month"] = df["settlement_date"].dt.month
df["day_of_week"] = df["settlement_date"].dt.dayofweek
df["is_weekend"] = df["day_of_week"].isin([5,6]).astype(int)

# Capacity utilization ratios
df["wind_utilization"] = df["embedded_wind_generation"] / (df["embedded_wind_capacity"] + 1)
df["solar_utilization"] = df["embedded_solar_generation"] / (df["embedded_solar_capacity"] + 1)

# Net cross-border flow sum
flow_cols = ["ifa2_flow","britned_flow","moyle_flow","east_west_flow","nemo_flow"]
df["net_crossborder_flow"] = df[flow_cols].sum(axis=1)

In [33]:
df = df.sample(50000, random_state=SEED)  # 50k rijen

In [35]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.3,
    random_state=SEED,
    shuffle=True
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=1/3,
    random_state=SEED,
    shuffle=True
)

In [ ]:
reg = setup(
    data=df,
    target=target,
    session_id=SEED,
    normalize=True,
    polynomial_features=False,
    transform_target=True,
    remove_multicollinearity=True,
    multicollinearity_threshold=0.9,
    use_gpu=False,
    html=False
)

                    Description                 Value
0                    Session id                   128
1                        Target  england_wales_demand
2                   Target type            Regression
3           Original data shape           (35000, 23)
4        Transformed data shape           (35000, 23)
5   Transformed train set shape           (24500, 23)
6    Transformed test set shape           (10500, 23)
7              Numeric features                    21
8                 Date features                     1
9                    Preprocess                  True
10              Imputation type                simple
11           Numeric imputation                  mean
12       Categorical imputation                  mode
13     Remove multicollinearity                  True
14  Multicollinearity threshold                   0.9
15                    Normalize                  True
16             Normalize method                zscore
17             Transform tar

In [29]:
best_model = compare_models(sort="MAE", n_select=1, turbo=True)

                                    Model        MAE           MSE       RMSE  \
et                  Extra Trees Regressor   117.3018  4.229297e+04   197.5521   
rf                Random Forest Regressor   121.8199  4.397214e+04   201.9744   
lightgbm  Light Gradient Boosting Machine   125.3872  4.496488e+04   205.2193   
gbr           Gradient Boosting Regressor   151.9255  5.444072e+04   229.5941   
dt                Decision Tree Regressor   174.2538  7.777614e+04   274.7552   
huber                     Huber Regressor   231.5897  1.054922e+05   322.0898   
ridge                    Ridge Regression   233.2211  1.047435e+05   320.9263   
br                         Bayesian Ridge   233.2231  1.047469e+05   320.9318   
lar                Least Angle Regression   233.2231  1.047471e+05   320.9321   
lr                      Linear Regression   233.2300  1.047265e+05   320.8993   
lasso                    Lasso Regression   245.1897  1.122877e+05   332.6282   
llar         Lasso Least Ang

In [30]:
tuned_model = tune_model(
    best_model, 
    optimize="MAE", 
    fold=5,
    n_iter=20
)


Processing:   0%|          | 0/7 [00:00<?, ?it/s]

Fitting 5 folds for each of 20 candidates, totalling 100 fits


KeyboardInterrupt: 

In [ ]:
final_model = finalize_model(tuned_model)
save_model(final_model, "england_wales_demand_predictor")